# 60 CNN GNN LSTM Transformer

Hybrid LSTM/Transformer trajectory model using the same saved split as every other notebook.


In [1]:
import gc
import json
import sys
from pathlib import Path

import torch

gc.collect()
try:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
except Exception:
    pass

PROJECT_ROOT = Path.home() / "Documents/Thesis"
PIPELINE_ROOT = PROJECT_ROOT / "model_training"
if str(PIPELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(PIPELINE_ROOT))

from training.notebook_workflow import (
    CNNGNNLSTMTransformerTrajectoryPredictor,
    ScanGraphTrajectoryDataset,
    collate_scan_graph,
    describe_dataloaders,
    describe_model,
    describe_shared_artifacts,
    device_from_flag,
    evaluate_trajectory_model,
    finish_runtime_report,
    load_or_build_shared_artifacts,
    make_dataloaders,
    prepare_result_dirs,
    save_final_trajectory_evaluation,
    start_runtime_report,
    set_seed,
    timestamp_tag,
    train_trajectory_model,
)


In [2]:
SEED = 42
PAST_LEN = 10
FUTURE_LEN = 5
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
MAX_SAMPLES = None
USE_CPU = False

BATCH_SIZE = 64
EPOCHS = 30
EARLY_STOPPING_PATIENCE = 5
LR = 1e-3
WEIGHT_DECAY = 1e-4

GOAL_DIM = 13
NODE_DIM = 12
EDGE_DIM = 8
HIDDEN_DIM = 128
GRAPH_HIDDEN = 96
DROPOUT = 0.10
MSG_PASSES = 2
TRANSFORMER_HEADS = 4
TRANSFORMER_LAYERS = 2
TRANSFORMER_FF = 256

device = device_from_flag(USE_CPU)
print("Device:", device)


Device: cuda


In [3]:
shared_runtime = start_runtime_report(
    stage_name="shared_data_split",
    output_dir=PIPELINE_ROOT / "results",
    context={"past_len": PAST_LEN, "future_len": FUTURE_LEN, "seed": SEED},
)
set_seed(SEED)
streams, sample_table, split_info, sample_table_path, split_path = load_or_build_shared_artifacts(
    past_len=PAST_LEN,
    future_len=FUTURE_LEN,
    seed=SEED,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
)
print("Sample table:", sample_table_path)
print("Split path:", split_path)
print("Split strategy:", split_info["split_strategy"])
print("Episode count:", split_info["episode_count"])
print("Train / Val / Test samples:", len(split_info["train_indices"]), len(split_info["val_indices"]), len(split_info["test_indices"]))
print("Train episodes:", split_info["train_episode_ids"])
print("Val episodes:", split_info["val_episode_ids"])
print("Test episodes:", split_info["test_episode_ids"])
shared_runtime_summary = finish_runtime_report(
    shared_runtime,
    extra=describe_shared_artifacts(
        streams=streams,
        sample_table=sample_table,
        split_info=split_info,
        sample_table_path=sample_table_path,
        split_path=split_path,
    ),
)
print(json.dumps(shared_runtime_summary, indent=2))


[runtime] shared_data_split start: 2026-05-23T12:08:10 summary=/home/basudeo/Documents/Thesis/model_training/results/runtime/20260523_120810_shared_data_split_runtime_summary.json
Sample table: /home/basudeo/Documents/Thesis/model_training/results/train_ready/sample_table_seed42_past10_future5.json
Split path: /home/basudeo/Documents/Thesis/model_training/results/splits/trajectory_split_seed42_past10_future5.json
Split strategy: episode
Episode count: 8
Train / Val / Test samples: 36405 6082 9219
Train episodes: ['run_model_20260522_203433', 'run_model_20260522_204215', 'run_model_20260523_013946', 'run_model_20260523_015017', 'run_model_20260522_202537']
Val episodes: ['run_model_20260523_013116']
Test episodes: ['run_model_20260522_201207', 'run_model_20260522_201854']
[runtime] shared_data_split done: start=2026-05-23T12:08:10 end=2026-05-23T12:08:15 elapsed_s=4.334 peak_cpu=11.10% peak_rss_mb=1046.43 peak_gpu_alloc_mb=0.00
{
  "stage_name": "shared_data_split",
  "timestamp": "2026

In [4]:
MODEL_SLUG = "cnn_gnn_lstm_transformer"
TIMESTAMP = timestamp_tag()
result_dir, weight_dir, plot_dir = prepare_result_dirs(MODEL_SLUG)

dataset = ScanGraphTrajectoryDataset(streams, sample_table, PAST_LEN)
train_loader, val_loader, test_loader = make_dataloaders(
    dataset,
    split_info,
    batch_size=BATCH_SIZE,
    collate_fn=collate_scan_graph,
    max_samples=MAX_SAMPLES,
)
loader_stats = describe_dataloaders(train_loader=train_loader, val_loader=val_loader, test_loader=test_loader)

model = CNNGNNLSTMTransformerTrajectoryPredictor(
    goal_dim=GOAL_DIM,
    node_dim=NODE_DIM,
    edge_dim=EDGE_DIM,
    hidden_dim=HIDDEN_DIM,
    graph_hidden=GRAPH_HIDDEN,
    future_len=FUTURE_LEN,
    dropout=DROPOUT,
    msg_passes=MSG_PASSES,
    num_heads=TRANSFORMER_HEADS,
    num_layers=TRANSFORMER_LAYERS,
    ff_dim=TRANSFORMER_FF,
).to(device)
model_stats = describe_model(model)
notebook_runtime = start_runtime_report(
    stage_name=f"{MODEL_SLUG}_notebook_run",
    output_dir=result_dir,
    context={
        "model_slug": MODEL_SLUG,
        "timestamp": TIMESTAMP,
        "past_len": PAST_LEN,
        "future_len": FUTURE_LEN,
        **loader_stats,
        **model_stats,
    },
)

train_out = train_trajectory_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    model_slug=MODEL_SLUG,
    timestamp=TIMESTAMP,
    split_path=split_path,
    sample_table_path=sample_table_path,
    result_dir=result_dir,
    weight_dir=weight_dir,
    plot_dir=plot_dir,
    epochs=EPOCHS,
    patience=EARLY_STOPPING_PATIENCE,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    extra_manifest={"family": "scan_graph_lstm_transformer", "future_len": FUTURE_LEN, "msg_passes": MSG_PASSES},
    runtime_output_dir=result_dir,
    runtime_context={
        **loader_stats,
        **model_stats,
        **describe_shared_artifacts(
            streams=streams,
            sample_table=sample_table,
            split_info=split_info,
            sample_table_path=sample_table_path,
            split_path=split_path,
        ),
    },
)
test_eval = evaluate_trajectory_model(model, test_loader, device)
notebook_runtime_summary = finish_runtime_report(
    notebook_runtime,
    extra={
        **loader_stats,
        **model_stats,
        "test_sample_count": int(len(test_loader.dataset)),
    },
)
final_metrics = save_final_trajectory_evaluation(
    model_slug=MODEL_SLUG,
    timestamp=TIMESTAMP,
    train_out=train_out,
    test_eval=test_eval,
    split_path=split_path,
    sample_table_path=sample_table_path,
    result_dir=result_dir,
    plot_dir=plot_dir,
    notebook_runtime=notebook_runtime_summary,
)
print(json.dumps(final_metrics, indent=2))


[runtime] cnn_gnn_lstm_transformer_notebook_run start: 2026-05-23T12:08:15 summary=/home/basudeo/Documents/Thesis/model_training/results/cnn_gnn_lstm_transformer/runtime/20260523_120815_cnn_gnn_lstm_transformer_notebook_run_runtime_summary.json
[runtime] cnn_gnn_lstm_transformer_training start: 2026-05-23T12:08:15 summary=/home/basudeo/Documents/Thesis/model_training/results/cnn_gnn_lstm_transformer/runtime/20260523_120815_cnn_gnn_lstm_transformer_training_runtime_summary.json
[cnn_gnn_lstm_transformer] start: epochs=30 patience=5 train_batches=569 val_batches=96
[cnn_gnn_lstm_transformer] epoch 01/30 train_loss=0.000658 val_loss=0.000044 val_ADE=0.009072 val_FDE=0.006783 val_RMSE=0.013275 best_ADE=0.009072 status=improved
[cnn_gnn_lstm_transformer] epoch 02/30 train_loss=0.000526 val_loss=0.000042 val_ADE=0.008589 val_FDE=0.005919 val_RMSE=0.013011 best_ADE=0.008589 status=improved
[cnn_gnn_lstm_transformer] epoch 03/30 train_loss=0.000515 val_loss=0.000043 val_ADE=0.008618 val_FDE=0.